# A Parser for Regular Expressions

This recursive top-down parser follows the same EBNF grammar as the Python notebook:
```
regExp  → product ('+' product)*
product → factor factor*
factor  → atom '*'*
atom    → '(' regExp ')' | LETTER | '𝜀' | '∅'
```
The parse tree is a `RegExp` discriminated union: `Empty`, `Epsilon`, `Literal c`, `Concat (r1,r2)`, `Alt (r1,r2)`, and `Star r`. Those cases retain the mathematical structure while avoiding tuples containing unchecked operator strings.

In [ ]:
// Checks remain active in both F# Interactive and the Jupyter kernel.
let checkNotebook condition =
    if not condition then failwith "Notebook check failed"


In [ ]:
#load "AutomataTypes.fsx"

In [ ]:
open AutomataTypes

## Tokenization
Each token has a distinct union case. Spaces are ignored; unexpected characters are reported instead of silently discarded. Both `𝜀` and `ε` denote the empty word. Only ASCII letters denote input symbols, matching the original tokenizer.

In [ ]:
type Token = Plus | Times | LeftParen | RightParen | Letter of char | EmptyToken | EpsilonToken

let tokenize (text: string) : Token list =
    // The lecture's mathematical italic epsilon occupies a UTF-16 surrogate pair.
    // Normalize it before scanning chars, and accept the ordinary epsilon as well.
    text.Replace("𝜀", "ε")
    |> Seq.choose (function
        | '+' -> Some Plus | '*' -> Some Times
        | '(' -> Some LeftParen | ')' -> Some RightParen
        | '∅' -> Some EmptyToken | 'ε' -> Some EpsilonToken
        | c when System.Char.IsWhiteSpace c -> None
        | c when ('a' <= c && c <= 'z') || ('A' <= c && c <= 'Z') -> Some (Letter c)
        | c -> invalidArg "text" (sprintf "Unexpected character '%c' in regular expression." c))
    |> Seq.toList

In [ ]:
tokenize "a*bc + ba*c + (𝜀+c*) + ∅*"

## The four grammar rules
`let rec ... and ...` declares mutually recursive functions together; F# needs no forward stubs. Each returns a parse tree together with the remaining tokens. `parseRegExp` accumulates alternatives, `parseProduct` concatenations, `parseFactor` repeated stars, and `parseAtom` the base cases. The alternatives and products associate to the left, just as in the original notebook.

In [ ]:
let rec parseRegExp tokens =
    let first, rest = parseProduct tokens
    let rec alternatives expression remaining =
        match remaining with
        | Plus :: tail ->
            let next, after = parseProduct tail
            alternatives (Alt (expression, next)) after
        | _ -> expression, remaining
    alternatives first rest

and parseProduct tokens =
    let first, rest = parseFactor tokens
    let rec factors expression remaining =
        match remaining with
        | (Letter _ | LeftParen | EmptyToken | EpsilonToken) :: _ ->
            let next, after = parseFactor remaining
            factors (Concat (expression, next)) after
        | _ -> expression, remaining
    factors first rest

and parseFactor tokens =
    let first, rest = parseAtom tokens
    let rec stars expression remaining =
        match remaining with
        | Times :: tail -> stars (Star expression) tail
        | _ -> expression, remaining
    stars first rest

and parseAtom tokens =
    match tokens with
    | LeftParen :: tail ->
        let expression, rest = parseRegExp tail
        match rest with
        | RightParen :: after -> expression, after
        | _ -> invalidArg "tokens" "Expected ')' after regular expression."
    | EmptyToken :: rest -> Empty, rest
    | EpsilonToken :: rest -> Epsilon, rest
    | Letter c :: rest -> Literal c, rest
    | _ -> invalidArg "tokens" "Expected a letter, ∅, epsilon, or parenthesized expression."

The entry point tokenizes the input, parses its expression, and requires complete consumption.

In [ ]:
let parse (text: string) : RegExp =
    let expression, rest = parseRegExp (tokenize text)
    if not (List.isEmpty rest) then invalidArg "text" (sprintf "Unexpected trailing tokens: %A" rest)
    expression

In [ ]:
parse "a*"

In [ ]:
parse "a**"

In [ ]:
parse "ab"

In [ ]:
parse "abc"

In [ ]:
parse "a+b+c"

In [ ]:
parse "ab+cd*"

In [ ]:
parse "(a+b*)*"

In [ ]:
parse "(a**+b*)*"

Malformed input is a parse error, including empty input, a missing operand or parenthesis, and unknown punctuation.

In [ ]:
for text in [""; "a+"; "(a"; "a)"; "a#"] do
    let rejected = try parse text |> ignore; false with :? System.ArgumentException -> true
    checkNotebook rejected